In [2]:
import pandas as pd

In [3]:
plans = pd.read_csv("../data/plans.csv")
claims = pd.read_csv("../data/claims.csv")

In [ ]:
plans.info()

In [ ]:
plans.head()

In [ ]:
claims.info()

In [ ]:
claims.head()

In [ ]:
claims["date_filed"] = pd.to_datetime(
    claims["date_filed"],
    errors="coerce"
)


In [ ]:
claims["date_filed"].dtype

In [ ]:
claims.info()


In [ ]:
print("Plans duplicates:", plans.duplicated().sum())
print("Claims duplicates:", claims.duplicated().sum())


In [ ]:
plans = plans.drop_duplicates()
claims = claims.drop_duplicates()


In [ ]:
print("Plans nulls:")
print(plans.isnull().sum())

print("\nClaims nulls:")
print(claims.isnull().sum())

In [6]:
import sqlite3

In [7]:
conn = sqlite3.connect("../coverage.db")

In [8]:
plans.to_sql("plans", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)


5

In [18]:
pd.read_sql("SELECT * FROM plans", conn)

,plan_id,plan_name,monthly_premium,annual_deductible,copay_pct,coverage_type,network_tier
0,P101,Gold PPO,500,2000,10,PPO,Gold
1,P102,Silver HMO,300,1500,20,HMO,Silver
2,P103,Bronze HMO,150,1000,30,HMO,Bronze


In [17]:
pd.read_sql("SELECT * FROM claims", conn)

,claim_id,member_id,plan_id,procedure,claim_amount,status,date_filed
0,C1001,M1001,P101,X-ray,250,Pending,2023-04-01
1,C1002,M1001,P101,Surgery,1200,Approved,2023-03-15
2,C1003,M1002,P102,X-ray,150,Denied,2023-04-05
3,C1004,M1002,P102,Surgery,900,Approved,2023-03-20
4,C1005,M1003,P103,X-ray,50,Pending,2023-04-10


In [16]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

,name
0,plans
1,claims


In [ ]:
conn.close()

In [9]:
pd.read_sql("""
    SELECT plan_name, annual_deductible
    FROM plans
    WHERE plan_name = 'Gold PPO';
""", conn)

,plan_name,annual_deductible
0,Gold PPO,2000


In [10]:
pd.read_sql("""
    SELECT COUNT(*) AS pending_claims
    FROM claims
    WHERE member_id = 'M1001'
      AND status = 'Pending';
""", conn)

,pending_claims
0,1


In [11]:
pd.read_sql("""
    SELECT plan_name, monthly_premium
    FROM plans
    WHERE monthly_premium < 400;
""", conn)

,plan_name,monthly_premium
0,Silver HMO,300
1,Bronze HMO,150


In [12]:
pd.read_sql("""
    SELECT
        c.claim_id,
        c.member_id,
        c.procedure,
        c.claim_amount,
        c.status,
        p.plan_name,
        p.coverage_type
    FROM claims c
    JOIN plans p
        ON c.plan_id = p.plan_id;
""", conn)

,claim_id,member_id,procedure,claim_amount,status,plan_name,coverage_type
0,C1001,M1001,X-ray,250,Pending,Gold PPO,PPO
1,C1002,M1001,Surgery,1200,Approved,Gold PPO,PPO
2,C1003,M1002,X-ray,150,Denied,Silver HMO,HMO
3,C1004,M1002,Surgery,900,Approved,Silver HMO,HMO
4,C1005,M1003,X-ray,50,Pending,Bronze HMO,HMO


In [13]:
pd.read_sql("""
    SELECT
        procedure,
        COUNT(*) AS claim_count
    FROM claims
    GROUP BY procedure
    ORDER BY claim_count DESC
    LIMIT 2;
""", conn)

,procedure,claim_count
0,X-ray,3
1,Surgery,2
